DATA INGESTION PIPELINE- From data ingestion to vector db

In [23]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [24]:
def process_all_pdf(pdf_directory):
    #Aim is to process all the pdf's in the directory
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    #Find all the pdf files only
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for i in pdf_files:
        print(f"Processing file: {i.name}")
        try:
            loader=PyPDFLoader(str(i))
            documents=loader.load()
            for doc in documents:
                doc.metadata["source_file"]=i.name
                doc.metadata["file_type"]="pdf"
            all_documents.extend(documents)
            print(f"Successfully processed {i.name} with PyPDFLoader.")
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"Total pages loaded: {len(all_documents)}")
    return all_documents

all_docs=process_all_pdf("../data")

Found 3 PDF files in the directory.
Processing file: 10.Numerical_Bert Model.pdf
Successfully processed 10.Numerical_Bert Model.pdf with PyPDFLoader.
Loaded 6 pages
Processing file: 6.RNN.pdf
Successfully processed 6.RNN.pdf with PyPDFLoader.
Loaded 23 pages
Processing file: 8.LanguageModels.pdf
Successfully processed 8.LanguageModels.pdf with PyPDFLoader.
Loaded 55 pages
Total pages loaded: 84


In [25]:
all_docs

[Document(metadata={'producer': 'Microsoft® OneNote® for Microsoft 365', 'creator': 'Microsoft® OneNote® for Microsoft 365', 'creationdate': '2026-04-02T17:28:05+05:30', 'moddate': '2026-04-02T17:28:05+05:30', 'source': '..\\data\\pdf\\10.Numerical_Bert Model.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': '10.Numerical_Bert Model.pdf', 'file_type': 'pdf'}, page_content='Feature BERT GPT (LLMs)\nDeveloped by Google OpenAI\nArchitecture Encoder-only Transformer Decoder-only Transformer\nContext understanding Bidirectional (← →) Unidirectional (left → right)\nTraining Objective Masked Language Model (MLM) Next Word Prediction\nUse case Understanding tasks Generation tasks\nExample Classification Chatbots, text generation\nBidirectional meaning in BERT\nBERT reads a word using both left and right context simultaneously.•\nExample (Very Important)\nSentence:\nHe sat on the bank of the river\nFocus word: bank\nHow BERT understands:\nLeft context → He sat on the•\nRight 

In [26]:
#Text splitting or basically splitting the documents into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):   
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
        )
    
    split_docs=text_splitter.split_documents(documents)#built in function split_documents
    print(f"split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
      print(f"Example chunk content:\n{split_docs[0].page_content[:500]}...") 
      print(f"Metadata of the chunk:\n{split_docs[0].metadata}")

    return split_docs    
    

In [27]:
chunks=split_documents(all_docs)
chunks

split 84 documents into 92 chunks.
Example chunk content:
Feature BERT GPT (LLMs)
Developed by Google OpenAI
Architecture Encoder-only Transformer Decoder-only Transformer
Context understanding Bidirectional (← →) Unidirectional (left → right)
Training Objective Masked Language Model (MLM) Next Word Prediction
Use case Understanding tasks Generation tasks
Example Classification Chatbots, text generation
Bidirectional meaning in BERT
BERT reads a word using both left and right context simultaneously.•
Example (Very Important)
Sentence:
He sat on the ban...
Metadata of the chunk:
{'producer': 'Microsoft® OneNote® for Microsoft 365', 'creator': 'Microsoft® OneNote® for Microsoft 365', 'creationdate': '2026-04-02T17:28:05+05:30', 'moddate': '2026-04-02T17:28:05+05:30', 'source': '..\\data\\pdf\\10.Numerical_Bert Model.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': '10.Numerical_Bert Model.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® OneNote® for Microsoft 365', 'creator': 'Microsoft® OneNote® for Microsoft 365', 'creationdate': '2026-04-02T17:28:05+05:30', 'moddate': '2026-04-02T17:28:05+05:30', 'source': '..\\data\\pdf\\10.Numerical_Bert Model.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': '10.Numerical_Bert Model.pdf', 'file_type': 'pdf'}, page_content='Feature BERT GPT (LLMs)\nDeveloped by Google OpenAI\nArchitecture Encoder-only Transformer Decoder-only Transformer\nContext understanding Bidirectional (← →) Unidirectional (left → right)\nTraining Objective Masked Language Model (MLM) Next Word Prediction\nUse case Understanding tasks Generation tasks\nExample Classification Chatbots, text generation\nBidirectional meaning in BERT\nBERT reads a word using both left and right context simultaneously.•\nExample (Very Important)\nSentence:\nHe sat on the bank of the river\nFocus word: bank\nHow BERT understands:\nLeft context → He sat on the•\nRight 

In [28]:
##embedding and vector store db
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple, Any
from sklearn.metrics.pairwise import cosine_similarity


In [29]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 930.65it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\JAYA AGRWAL\AppData\Local\Temp\ipykernel_25752\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [30]:
import os
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 55


In [31]:
chunks

[Document(metadata={'producer': 'Microsoft® OneNote® for Microsoft 365', 'creator': 'Microsoft® OneNote® for Microsoft 365', 'creationdate': '2026-04-02T17:28:05+05:30', 'moddate': '2026-04-02T17:28:05+05:30', 'source': '..\\data\\pdf\\10.Numerical_Bert Model.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': '10.Numerical_Bert Model.pdf', 'file_type': 'pdf'}, page_content='Feature BERT GPT (LLMs)\nDeveloped by Google OpenAI\nArchitecture Encoder-only Transformer Decoder-only Transformer\nContext understanding Bidirectional (← →) Unidirectional (left → right)\nTraining Objective Masked Language Model (MLM) Next Word Prediction\nUse case Understanding tasks Generation tasks\nExample Classification Chatbots, text generation\nBidirectional meaning in BERT\nBERT reads a word using both left and right context simultaneously.•\nExample (Very Important)\nSentence:\nHe sat on the bank of the river\nFocus word: bank\nHow BERT understands:\nLeft context → He sat on the•\nRight 

In [32]:
##converting the chunks to the embeddings
texts=[doc.page_content for doc in chunks]
texts

['Feature BERT GPT (LLMs)\nDeveloped by Google OpenAI\nArchitecture Encoder-only Transformer Decoder-only Transformer\nContext understanding Bidirectional (← →) Unidirectional (left → right)\nTraining Objective Masked Language Model (MLM) Next Word Prediction\nUse case Understanding tasks Generation tasks\nExample Classification Chatbots, text generation\nBidirectional meaning in BERT\nBERT reads a word using both left and right context simultaneously.•\nExample (Very Important)\nSentence:\nHe sat on the bank of the river\nFocus word: bank\nHow BERT understands:\nLeft context → He sat on the•\nRight context → of the river•\nSo BERT concludes:\nbank = river side (not financial bank)\nBERT processes:\n← full context + word + full context →\nSo meaning is context-aware and accurate\nCompare with Unidirectional Model\nIf model reads only left → right:\nHe sat on the bank\nIt may think:\nbank can be misunderstood as financial institution \nBecause it has not seen “river” yet\nWhen to Use BE

In [33]:
embeddings=embedding_manager.generate_embeddings(texts)
embeddings

Generating embeddings for 92 texts...


Batches: 100%|██████████| 3/3 [00:06<00:00,  2.03s/it]

Generated embeddings with shape: (92, 384)


array([[-0.06500881, -0.08050089, -0.01599829, ...,  0.09836325,
         0.00425841,  0.01861527],
       [-0.03889041, -0.07948749, -0.00139807, ...,  0.03158609,
         0.01418237,  0.00987078],
       [-0.16322547, -0.04764038,  0.05609919, ...,  0.03590714,
         0.00681492,  0.05451635],
       ...,
       [-0.06382501, -0.06817459,  0.04942302, ..., -0.0724097 ,
        -0.03981076,  0.05565909],
       [-0.02676572, -0.07559374,  0.05048631, ..., -0.03824417,
        -0.0112164 ,  0.04270991],
       [-0.10215606, -0.04578762,  0.07603401, ..., -0.04260054,
        -0.0358316 ,  0.03267813]], shape=(92, 384), dtype=float32)

In [34]:
##storing in vector db
vectorstore.add_documents(chunks,embeddings)

Adding 92 documents to vector store...
Successfully added 92 documents to vector store
Total documents in collection: 147


In [35]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [36]:
rag_retriever.retrieve("What is rnn?")

Retrieving documents for query: 'What is rnn?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a99be4a4_11',
  'content': 'What is RNN ?  \nRecurrent Neural Network(RNN) is a type of Neural Network where \nthe output from the previous step is fed as input to the current step.  \n• In some cases when it is required to  predict the next word \nof a sentence, the previous words are required and \nhence there is a need to remember the previous \nwords. Thus RNN came into existence, which solved this \nissue with the help of a Hidden Layer.  \n• The main and most important feature of RNN is its  Hidden \nstate, which remembers some information about a sequence. \nThe state is also referred to as  Memory State since it \nremembers the previous input to the network.',
  'metadata': {'creationdate': 'D:20250609131017',
   'content_length': 635,
   'doc_index': 11,
   'creator': 'PDFium',
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\6.RNN.pdf',
   'page': 0,
   'producer': 'PDFium',
   'source_file': '6.RNN.pdf',
   'page_label': '1',
   'total_pages': 23},
  'similar

In [39]:
rag_retriever.retrieve("What is a language model?")

Retrieving documents for query: 'What is a language model?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_248818f8_9',
  'content': '• Our goal is to compute the probability of a sentence or sequence of words \nW (=w1,w2,…,wn):\nP(W) = P(w1,w2,…,wn)\n• What is the probability of an upcoming word?:\n– P(w5|w1,w2,w3,w4)\n• A model that computes either of these:\n– P(W)     or     P(wn|w1,w2…wn-1)          is called a language model.\nProbabilistic Language Models\nNatural Language Processing 10',
  'metadata': {'page': 9,
   'creator': 'Microsoft® PowerPoint® 2019',
   'creationdate': '2025-03-10T14:20:48+03:00',
   'source_file': '8.LanguageModels.pdf',
   'title': 'N-gram Language Models',
   'page_label': '10',
   'author': 'Ilyas Cicekli',
   'moddate': '2025-03-10T14:20:48+03:00',
   'total_pages': 55,
   'source': '..\\data\\pdf\\8.LanguageModels.pdf',
   'doc_index': 9,
   'file_type': 'pdf',
   'producer': 'Microsoft® PowerPoint® 2019',
   'content_length': 356},
  'similarity_score': 0.051609158515930176,
  'distance': 0.9483908414840698,
  'rank': 1},
 {'id': 'doc_4ea4

RAG PIPELINE WITH THE INTEGRATION OF LLM

In [45]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [46]:
answer=rag_simple("What is rnn?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is rnn?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


RNN (Recurrent Neural Network) is a type of neural network designed for sequential data. Unlike feed‑forward networks, it feeds the output (or hidden state) from the previous time step back into the network as input for the current step, allowing it to retain and use information from earlier elements in the sequence. This hidden/memory state lets RNNs model dependencies such as previous words in a sentence or past values in a time series, enabling tasks like next‑word prediction or time‑series forecasting.


ENHANCING THE RAG PIPELINE

In [49]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("RNN working", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'RNN working'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **How an RNN works**

1. **Sequential processing** – The network receives one element of a sequence at a time (e.g., a word, a time‑step).  

2. **Hidden (memory) state** – At each step *t* the RNN keeps a hidden vector **hₜ** that summarizes everything seen so far.  

3. **Recurrence equation**  
   \[
   h_t = f\big(W_{xh}x_t + W_{hh}h_{t-1} + b_h\big)
   \]
   * *xₜ* – current input vector  
   * *Wₓₕ* – input‑to‑hidden weights  
   * *Wₕₕ* – recurrent (hidden‑to‑hidden) weights  
   * *f* – non‑linear activation (tanh, ReLU, etc.)  

4. **Output generation**  
   \[
   y_t = g\big(W_{hy}h_t + b_y\big)
   \]
   where *g* is usually a soft‑max (for classification) or a linear map (for regression).  

5. **Parameter sharing** – The same weight matrices (*Wₓₕ*, *Wₕₕ*, *W_{hy}*) are used at every time step, which lets the model generalize across sequence lengths.  

6. **Training** – The whole unrolled network (one copy per time step) is trained by back‑propagation through time 

In [50]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)



Final Answer: No relevant context found.
Summary: The response indicates that there is no pertinent information available. It suggests that the necessary context is missing.
History: {'question': 'what is attention is all you need', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'The response indicates that there is no pertinent information available. It suggests that the necessary context is missing.'}
